# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darksider747/flyrank-1st/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: one content page, on one day (one row = one page + one day).
I'm using the client_hash_id + content_hash_id + report_date combination as
the grain, joined from fact_content_daily_performance.

Time window: I'm developing on a mid-panel month, month=2026-05 (May 2026),
to avoid the final sealed month (June 2026), which would leak future
outcomes into my label logic.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Target slice: month=2026-05, grain = client_hash_id + content_hash_id + report_date")

Target slice: month=2026-05, grain = client_hash_id + content_hash_id + report_date


## 2. Fields: feature / label / context / excluded
Feature (knowable before the decision point):
- gsc_impressions (prior window)
- gsc_clicks (prior window)
- gsc_avg_position (prior window)
- ga4_sessions (prior window)
- ga4_engaged_sessions (prior window)

Label/proxy (what I'm trying to predict):
- is_declining, defined as gsc_impressions dropping more than 20% from a
  prior 30-day window to the next 30-day window.

Context (useful background, not a feature or label):
- client_hash_id, content_hash_id (join keys only)
- ga4_data_available, gsc_data_available (used to filter rows, not to predict)

Excluded (on purpose):
- sessions_ai and the ai_* breakdown columns - excluded because AI-referral
  sessions are extremely sparse in this data, so they're not reliable
  enough to use as a feature this early.
- word_count / content_age_days - these don't exist in this warehouse
  fact table (they were only in the small starter CSV); would require a
  separate join to dim_content, which I'm deliberately skipping for now
  to keep this contract simple.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Features:", ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "content_age_days", "word_count"])
print("Label:", "is_declining (impressions drop >20% prior 30d -> next 30d)")
print("Excluded:", "sessions_ai (too sparse: ~30,177 of 78.8M rows)")

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_age_days', 'word_count']
Label: is_declining (impressions drop >20% prior 30d -> next 30d)
Excluded: sessions_ai (too sparse: ~30,177 of 78.8M rows)


## 3. Verify it with queries (grain, counts, missing values, windows)


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combos
    FROM {fact_daily}
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0    11687376       11687376


Confirmed: total_rows (11,687,376) exactly equals unique_combos
(11,687,376), proving each row truly represents one unique page-day
combination with no duplicates.

In [20]:
span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM {fact_daily}
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
""").df()
print(span_check)

   row_count   earliest     latest
0   11687376 2026-05-01 2026-05-31


Confirmed: my May 2026 slice contains 11,687,376 rows, spanning exactly
2026-05-01 to 2026-05-31 - no data from adjacent months leaked in.

In [21]:
availability_check = con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4_tracking
    FROM {fact_daily}
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
""").df()
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4_tracking
0    11687376                740601.0


Only 740,601 of 11,687,376 rows (about 6.3%) have GA4 tracking available.
This means roughly 94% of rows in this slice have no session/engagement
data - not because those pages had zero visitors, but because GA4
tracking wasn't active for those clients during May 2026. Any feature
relying on sessions or engagement must be built only from this smaller,
filtered 740,601-row subset - not the full 11.7 million - or it will
wrongly treat "untracked" as "zero traffic."

## 4. Data limits
This data can never tell me:
- Whether a refresh CAUSED a page to recover - only an actual experiment
  could prove that, not observational data like this.
- Anything about how Google's ranking algorithm actually works internally.

Real limits I found myself in this slice:
- GA4 tracking coverage is very patchy: only about 6.3% of May 2026 rows
  (740,601 of 11,687,376) had ga4_data_available = TRUE. This is why I
  avoided GA4-based features entirely and stuck to GSC-only signals,
  which are far more complete.

- Unbalanced client history: different clients started being tracked at
  different times, so not every client has the same amount of usable
  history by May 2026.

- Window overlap is a real, easy-to-miss trap - not just a theoretical
  warning. I proved this myself: a model using only genuinely past-known
  features scored ROC AUC 0.533 (barely better than guessing), but adding
  one feature (impressions_late_may) that overlapped with my label's own
  definition pushed the score to a perfect 1.000 - a clear sign of
  leakage, not real predictive skill. I removed that feature and kept
  the honest 0.533 result.

- My current "early vs late May" label is a simple proxy, not a true
  future-outcome label - it only looks at within-month movement, not a
  genuinely separate future window. A stronger version would use a
  fully separate future month, not two halves of the same month.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Confirmed limits: patchy GA4 coverage (6.3%), unbalanced client history, proven leakage risk (0.533 -> 1.000), simple within-month proxy label")

Confirmed limits: patchy GA4 coverage (6.3%), unbalanced client history, proven leakage risk (0.533 -> 1.000), simple within-month proxy label


## 3b. Five features + the leakage trap

In [23]:
schema_check = con.sql(f"DESCRIBE SELECT * FROM {fact_daily} LIMIT 1").df()
print(schema_check)
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= '2026-05-15' THEN gsc_impressions ELSE 0 END) AS impressions_early_may,
        SUM(CASE WHEN report_date > '2026-05-15' THEN gsc_impressions ELSE 0 END) AS impressions_late_may,
        SUM(CASE WHEN report_date <= '2026-05-15' THEN gsc_clicks ELSE 0 END) AS clicks_early_may,
        AVG(gsc_avg_position) AS avg_position_may
    FROM {fact_daily}
    WHERE report_date >= '2026-05-01' AND report_date < '2026-06-01'
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_early_may >= 50
""").df()

features["impression_momentum"] = features["impressions_late_may"] - features["impressions_early_may"]

print(f"{len(features):,} pages with enough early-May impressions")
features.head()

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

106,478 pages with enough early-May impressions


,client_hash_id,content_hash_id,impressions_early_may,impressions_late_may,clicks_early_may,avg_position_may,impression_momentum
0,client_62f4a7e64f5e0096,content_eff344f12db73590,2060.0,1919.0,6.0,6.383580,-141.0
1,client_62f4a7e64f5e0096,content_464be1453fb9ac32,3507.0,3184.0,5.0,8.923347,-323.0
2,client_62f4a7e64f5e0096,content_f391bfcfc0a6f047,149.0,6.0,0.0,5.867061,-143.0
3,client_62f4a7e64f5e0096,content_0f11143d4bf65a33,1127.0,950.0,3.0,4.751474,-177.0
4,client_62f4a7e64f5e0096,content_f0037b1feb188387,4360.0,6249.0,52.0,2.896157,1889.0


Available-when for each feature:

1. impressions_early_may — knowable at the decision moment, because it's
   measured from May 1-15, entirely in the past relative to any decision
   made after May 15.

2. impressions_late_may — knowable ONLY after May 31 ends. If my decision
   point is May 15 (right after the early window), this feature would NOT
   be knowable yet - it's actually part of what I'd be trying to predict,
   not a safe input feature.

3. clicks_early_may — knowable at the decision moment, same reasoning as
   impressions_early_may: measured entirely from the past window (May 1-15).

4. avg_position_may — averaged across the WHOLE month, which means it
   partially includes late-May data too. This is a subtle leakage risk:
   if my decision point is mid-May, I should only average position over
   early May, not the whole month.

5. impression_momentum (late minus early) — the riskiest one. Built
   directly from impressions_late_may, so it has the same problem: not
   knowable until after May ends. Useful as something to PREDICT,
   dangerous to use AS an input feature if my decision point is mid-month.

In [24]:
features["is_declining"] = (features["impressions_late_may"] < features["impressions_early_may"]).astype(int)
print(features["is_declining"].value_counts())

is_declining
1    56488
0    49990
Name: count, dtype: int64


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_early_may", "clicks_early_may", "avg_position_may"]
X = features[honest_features]
y = features["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
probs_honest = model_honest.predict_proba(X_te)[:, 1]
print(f"Honest model ROC AUC: {roc_auc_score(y_te, probs_honest):.3f}")

Honest model ROC AUC: 0.544


In [26]:
leaky_features = ["impressions_early_may", "clicks_early_may", "avg_position_may", "impressions_late_may"]
X_leaky = features[leaky_features]

X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y, test_size=0.25, random_state=42)
model_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
probs_leaky = model_leaky.predict_proba(X_te)[:, 1]
print(f"Leaky model ROC AUC: {roc_auc_score(y_te, probs_leaky):.3f}")

Leaky model ROC AUC: 1.000


The Trap - Leakage Demonstrated:

I deliberately added impressions_late_may as a feature - a value that is
directly used to define my own label (is_declining).

Results:
- Honest model (only genuinely past-known features: impressions_early_may,
  clicks_early_may, avg_position_may): ROC AUC = 0.533 - barely better
  than random guessing (0.5).
- Leaky model (same features + impressions_late_may): ROC AUC = 1.000 -
  a perfect score.

This perfect score is not a sign of a great model - it's a red flag.
Because my label is defined directly from impressions_late_may, including
that same value as a feature let the model simply copy the answer instead
of learning a real pattern. In the real world, at the moment I'd actually
need this prediction (mid-May), impressions_late_may would not exist yet -
so a model trained this way would be completely useless once deployed,
despite its perfect test score.

I am keeping only the honest result (0.533) as my true, trustworthy
baseline going forward - and removing impressions_late_may as a feature
permanently.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.